# Causal features and independent engines

## Objective
Verify temporal ordering, construct RUL and inspect the retained feature matrix.

## Method
Use the official FD001 files and saved engine-separated artifacts. Rebuild training with `python scripts/train_models.py` before rerunning if configuration changes.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'app.py').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
from src.data_loader import read_raw
from src.features import make_features, targets
train = read_raw()
test = read_raw('test')
results = json.loads((ROOT / 'artifacts/metrics/results.json').read_text())


In [2]:
y=targets(train)
x=make_features(train,results['sensors'])
print('Feature shape:',x.shape)
print('RUL range:',y.min(),y.max())
print('Training cap:',results['rul_cap'])
print(x.head(3).to_string(index=False))
a,b,c=[set(results['split'][k]) for k in ['train','validation','calibration']]
assert not a&b and not a&c and not b&c
print('Engine counts:',len(a),len(b),len(c))

Feature shape: (20631, 61)
RUL range: 0 361
Training cap: 125
 cycle     s2      s3      s4    s6     s7      s8      s9   s11    s12     s13     s14    s15  s17   s20     s21  s2_mean10  s2_std10  s2_delta5   s3_mean10  s3_std10  s3_delta5   s4_mean10  s4_std10  s4_delta5  s6_mean10  s6_std10  s6_delta5  s7_mean10  s7_std10  s7_delta5  s8_mean10  s8_std10  s8_delta5   s9_mean10  s9_std10  s9_delta5  s11_mean10  s11_std10  s11_delta5  s12_mean10  s12_std10  s12_delta5  s13_mean10  s13_std10  s13_delta5  s14_mean10  s14_std10  s14_delta5  s15_mean10  s15_std10  s15_delta5  s17_mean10  s17_std10  s17_delta5  s20_mean10  s20_std10  s20_delta5  s21_mean10  s21_std10  s21_delta5
     1 641.82 1589.70 1400.60 21.61 554.36 2388.06 9046.19 47.47 521.66 2388.02 8138.62 8.4195  392 39.06 23.4190 641.820000   0.00000        0.0 1589.700000  0.000000        0.0 1400.600000  0.000000        0.0      21.61       0.0        0.0 554.360000  0.000000        0.0    2388.06   0.00000        0.0 9046.1900

## Interpretation and conclusion
All windows use current and past values within one engine. The saved split excludes all shared engines between fit, selection and interval calibration.